# MediaPipe-33 ST-GAT Motion Predictor

This notebook trains an app-compatible Spatio-Temporal Graph Attention Transformer for Level 2.

Expected ONNX contract:
- input `past_landmarks`: `[1, 60, 33, 3]`
- output `future_landmarks`: `[1, 15, 33, 3]`

Normalization matches the web app: hip center origin and shoulder/torso scale.

In [ ]:
!pip -q install torch numpy matplotlib tqdm onnx onnxruntime

In [ ]:
import math, random, os, json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_JOINTS = 33
COORD_DIM = 3
PAST_FRAMES = 60
FUTURE_FRAMES = 15
STRIDE = 5
print('DEVICE:', DEVICE)

## Load Data

Upload `.npz` files containing pose sequences shaped `[T, J, C]` or `[N, T, J, C]` if available. If no files are uploaded, the notebook creates synthetic MediaPipe-like motion for smoke testing.

In [ ]:
from google.colab import files
uploaded = files.upload()
npz_files = [name for name in uploaded.keys() if name.endswith('.npz')]
print('NPZ files:', npz_files)

In [ ]:
def walk_arrays(obj):
    if isinstance(obj, dict):
        for value in obj.values(): yield from walk_arrays(value)
    elif isinstance(obj, (list, tuple)):
        for value in obj: yield from walk_arrays(value)
    elif isinstance(obj, np.ndarray):
        yield obj

def adapt_joint_count(seq, target_joints=33):
    seq = np.asarray(seq, dtype=np.float32)
    if seq.shape[-1] < 3:
        seq = np.pad(seq, [(0,0), (0,0), (0, 3 - seq.shape[-1])])
    seq = seq[..., :3]
    joints = seq.shape[1]
    if joints == target_joints:
        return seq
    if joints > target_joints:
        idx = np.linspace(0, joints - 1, target_joints).round().astype(int)
        return seq[:, idx, :]
    pad = np.repeat(seq[:, -1:, :], target_joints - joints, axis=1)
    return np.concatenate([seq, pad], axis=1)

def normalize_mediapipe33(seq):
    seq = adapt_joint_count(seq, NUM_JOINTS)
    left_hip, right_hip = seq[:, 23:24, :], seq[:, 24:25, :]
    root = (left_hip + right_hip) / 2.0
    shoulder_width = np.linalg.norm(seq[:, 11, :] - seq[:, 12, :], axis=-1, keepdims=True)
    shoulder_center = (seq[:, 11, :] + seq[:, 12, :]) / 2.0
    torso = np.linalg.norm(shoulder_center - root[:, 0, :], axis=-1, keepdims=True)
    scale = np.maximum(np.maximum(shoulder_width, torso), 1e-3)
    return np.nan_to_num((seq - root) / scale[:, None, :]).astype(np.float32)

def load_npz_sequences(paths, min_frames=90):
    sequences = []
    for path in paths:
        data = np.load(path, allow_pickle=True)
        for key in data.keys():
            obj = data[key]
            candidates = []
            if isinstance(obj, np.ndarray) and obj.dtype == object:
                try: candidates = list(walk_arrays(obj.item()))
                except Exception: candidates = []
            else:
                candidates = [obj]
            for arr in candidates:
                arr = np.asarray(arr)
                if arr.ndim == 3 and arr.shape[0] >= min_frames and arr.shape[-1] >= 2:
                    sequences.append(normalize_mediapipe33(arr))
                elif arr.ndim == 4 and arr.shape[1] >= min_frames and arr.shape[-1] >= 2:
                    for clip in arr:
                        if clip.shape[0] >= min_frames:
                            sequences.append(normalize_mediapipe33(clip))
    return sequences

def synthetic_mediapipe33(frames=180, motion_type=0, noise=0.01):
    base = np.zeros((33, 3), dtype=np.float32)
    base[23] = [-0.12, 0.0, 0]; base[24] = [0.12, 0.0, 0]
    base[11] = [-0.18, 0.55, 0]; base[12] = [0.18, 0.55, 0]
    base[13] = [-0.38, 0.35, 0]; base[14] = [0.38, 0.35, 0]
    base[15] = [-0.58, 0.18, 0]; base[16] = [0.58, 0.18, 0]
    base[25] = [-0.13, -0.45, 0]; base[26] = [0.13, -0.45, 0]
    base[27] = [-0.15, -0.88, 0]; base[28] = [0.15, -0.88, 0]
    base[0] = [0, 0.85, 0]; base[31] = [-0.17, -0.98, 0]; base[32] = [0.17, -0.98, 0]
    for i in range(33):
        if not np.any(base[i]): base[i] = base[0]
    t = np.linspace(0, 1, frames)
    seq = np.repeat(base[None], frames, axis=0).copy()
    amp = random.uniform(0.04, 0.22); speed = random.uniform(1.0, 3.0); phase = random.random() * 2 * math.pi
    if motion_type == 0:
        reach = amp * np.clip(np.sin(np.pi * t), 0, None)
        seq[:, 14, 0] += 0.35 * reach; seq[:, 16, 0] += 1.0 * reach
    elif motion_type == 1:
        guard = amp * np.sin(2 * math.pi * speed * t + phase)
        seq[:, 13, 1] += guard; seq[:, 14, 1] -= guard; seq[:, 15, 1] += guard; seq[:, 16, 1] -= guard
    elif motion_type == 2:
        down = amp * (1 - np.cos(2 * math.pi * t))
        seq[:, :, 1] -= down[:, None]
    else:
        sway = amp * np.sin(2 * math.pi * speed * t + phase)
        seq[:, :, 0] += sway[:, None]
    seq += np.random.normal(0, noise, seq.shape).astype(np.float32)
    return normalize_mediapipe33(seq)

sequences = load_npz_sequences(npz_files)
if not sequences:
    print('No compatible upload found; using synthetic fallback.')
    sequences = [synthetic_mediapipe33(random.randint(130, 220), i % 4) for i in range(1200)]
print('Sequences:', len(sequences), sequences[0].shape)

In [ ]:
class MotionForecastDataset(Dataset):
    def __init__(self, sequences, past=60, future=15, stride=5, max_windows=40000):
        self.sequences = sequences; self.items = []
        for si, seq in enumerate(sequences):
            for start in range(0, max(0, len(seq) - past - future + 1), stride):
                self.items.append((si, start))
                if len(self.items) >= max_windows: break
            if len(self.items) >= max_windows: break
        self.past = past; self.future = future
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        si, start = self.items[idx]
        seq = self.sequences[si]
        return torch.tensor(seq[start:start+self.past]), torch.tensor(seq[start+self.past:start+self.past+self.future])

ds = MotionForecastDataset(sequences, PAST_FRAMES, FUTURE_FRAMES, STRIDE)
train_len = int(len(ds) * 0.85); val_len = len(ds) - train_len
train_ds, val_ds = random_split(ds, [train_len, val_len], generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
print('Windows:', len(ds), 'Train:', len(train_ds), 'Val:', len(val_ds))

In [ ]:
class GraphAttentionLayer(nn.Module):
    def __init__(self, d_model, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        a, _ = self.attn(x, x, x, need_weights=False)
        x = self.norm1(x + a)
        return self.norm2(x + self.ff(x))

class STGATMotionPredictor(nn.Module):
    def __init__(self, num_joints=33, coord_dim=3, d_model=128, heads=4, temporal_layers=3, future_frames=15):
        super().__init__()
        self.num_joints = num_joints; self.coord_dim = coord_dim; self.future_frames = future_frames
        self.joint_embed = nn.Linear(coord_dim, d_model)
        self.joint_id = nn.Parameter(torch.randn(1, 1, num_joints, d_model) * 0.02)
        self.graph_attn = GraphAttentionLayer(d_model, heads)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=heads, dim_feedforward=d_model*4, dropout=0.1, batch_first=True, activation='gelu')
        self.temporal = nn.TransformerEncoder(enc, num_layers=temporal_layers)
        self.future_queries = nn.Parameter(torch.randn(1, future_frames, d_model) * 0.02)
        dec = nn.TransformerDecoderLayer(d_model=d_model, nhead=heads, dim_feedforward=d_model*4, dropout=0.1, batch_first=True, activation='gelu')
        self.decoder = nn.TransformerDecoder(dec, num_layers=2)
        self.out = nn.Linear(d_model, num_joints * coord_dim)
    def forward(self, x):
        B, T, J, C = x.shape
        h = self.joint_embed(x) + self.joint_id[:, :, :J, :]
        h = self.graph_attn(h.reshape(B*T, J, -1)).reshape(B, T, J, -1)
        memory = self.temporal(h.mean(dim=2))
        q = self.future_queries.repeat(B, 1, 1)
        return self.out(self.decoder(q, memory)).reshape(B, self.future_frames, self.num_joints, self.coord_dim)

model = STGATMotionPredictor(NUM_JOINTS, COORD_DIM, 128, 4, 3, FUTURE_FRAMES).to(DEVICE)
print('Params M:', round(sum(p.numel() for p in model.parameters()) / 1e6, 3))

In [ ]:
def mpjpe(pred, target): return torch.norm(pred - target, dim=-1).mean()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
EPOCHS = 10
best_val = 1e9
for epoch in range(1, EPOCHS + 1):
    model.train(); total = 0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch} train'):
        x = x.to(DEVICE).float(); y = y.to(DEVICE).float()
        optimizer.zero_grad(); pred = model(x); loss = mpjpe(pred, y)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        total += loss.item() * x.size(0)
    train_loss = total / len(train_loader.dataset)
    model.eval(); total = 0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f'Epoch {epoch} val'):
            x = x.to(DEVICE).float(); y = y.to(DEVICE).float()
            total += mpjpe(model(x), y).item() * x.size(0)
    val_loss = total / len(val_loader.dataset)
    print(f'Epoch {epoch}: train={train_loss:.5f}, val={val_loss:.5f}')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model_state': model.state_dict(), 'past_frames': PAST_FRAMES, 'future_frames': FUTURE_FRAMES, 'num_joints': NUM_JOINTS}, 'stgat_motion_predictor_mediapipe33.pt')
        print('Saved best')

In [ ]:
ckpt = torch.load('stgat_motion_predictor_mediapipe33.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state']); model.eval()
dummy = torch.randn(1, PAST_FRAMES, NUM_JOINTS, COORD_DIM).to(DEVICE)
onnx_path = 'stgat_motion_predictor.onnx'
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['past_landmarks'], output_names=['future_landmarks'],
    dynamic_axes={'past_landmarks': {0: 'batch'}, 'future_landmarks': {0: 'batch'}},
    opset_version=17
)
metadata = {'past_frames': PAST_FRAMES, 'future_frames': FUTURE_FRAMES, 'num_joints': NUM_JOINTS, 'coord_dim': COORD_DIM, 'normalization': 'hip_center_origin_shoulder_or_torso_scale'}
with open('stgat_motion_predictor_metadata.json', 'w') as f: json.dump(metadata, f, indent=2)
print('Exported', onnx_path, metadata)

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession('stgat_motion_predictor.onnx', providers=['CPUExecutionProvider'])
example = np.random.randn(1, PAST_FRAMES, NUM_JOINTS, COORD_DIM).astype(np.float32)
out = sess.run(None, {'past_landmarks': example})[0]
print('Input:', example.shape, 'Output:', out.shape)
assert out.shape == (1, FUTURE_FRAMES, NUM_JOINTS, COORD_DIM)
files.download('stgat_motion_predictor.onnx')
files.download('stgat_motion_predictor_metadata.json')